# Diffusion Transformer (DiT) Block with adaLN-Zero — Solution

**Difficulty**: 🔴 Hard

---

### Problem Statement

**DiT** (Peebles & Xie, *Scalable Diffusion Models with Transformers*, ICCV 2023) replaced the U-Net of latent diffusion with a plain Vision Transformer over patches of the VAE latent. The key design is how the conditioning vector $c$ (timestep embedding + class embedding) enters every block: **adaptive LayerNorm**. A small MLP regresses, from $c$, a per-sample *shift*, *scale* and *gate* for each residual branch; the LayerNorms themselves have no learnable affine parameters. In **adaLN-Zero** the regression layer is zero-initialized, so every gate starts at 0 and every block starts as the identity — the variant that won the paper's ablation against in-context tokens and cross-attention. DiT blocks (and their MMDiT descendants) are the backbone of PixArt, Stable Diffusion 3, Flux and Sora-style video models.

For each block, with $c \in \mathbb{R}^{B \times D}$ and tokens $x \in \mathbb{R}^{B \times N \times D}$:

$$(\beta_1, \gamma_1, \alpha_1, \beta_2, \gamma_2, \alpha_2) = \mathrm{Linear}(\mathrm{SiLU}(c)), \qquad \mathrm{modulate}(h, \beta, \gamma) = h \odot (1 + \gamma) + \beta$$

$$x \leftarrow x + \alpha_1 \odot \mathrm{Attn}\big(\mathrm{modulate}(\mathrm{LN}(x), \beta_1, \gamma_1)\big), \qquad x \leftarrow x + \alpha_2 \odot \mathrm{MLP}\big(\mathrm{modulate}(\mathrm{LN}(x), \beta_2, \gamma_2)\big)$$

where each $(B, D)$ vector is broadcast over the $N$ tokens of its sample. You will also write the timestep embedder and the `patchify` / `unpatchify` pair that turns latents into tokens and back.

---

### Requirements

1. **`modulate(x, shift, scale)`** — `x`: `(B, N, D)`, `shift`, `scale`: `(B, D)` → `x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)`.
2. **`TimestepEmbedder(hidden_size, freq_dim=256)`** — an `nn.Module` with
   - a `@staticmethod timestep_embedding(t, dim, max_period=10000)` → `(N, dim)` sinusoidal features of (possibly fractional or integer) timesteps `t` of shape `(N,)`: with $h$ = `dim // 2` and $P$ = `max_period`, frequencies $f_i = \exp(-\ln P \cdot i / h)$ for $i = 0, \dots, h-1$, arguments $a = t\,f$, and output `cat([cos(a), sin(a)], dim=-1)` (the DiT layout; assume `dim` even);
   - `forward(t)` = `Linear(freq_dim, hidden_size) → SiLU → Linear(hidden_size, hidden_size)` applied to `timestep_embedding(t, freq_dim)` → `(N, hidden_size)`.
3. **`DiTBlock(hidden_size, num_heads, mlp_ratio=4.0)`** with `forward(x, c)`, `x`: `(B, N, D)`, `c`: `(B, D)` → `(B, N, D)`:
   - `norm1`, `norm2` = `nn.LayerNorm(D, elementwise_affine=False, eps=1e-6)`;
   - multi-head self-attention over the $N$ tokens: `num_heads` heads of size `D // num_heads`, scores scaled by `1 / sqrt(head_dim)`, q/k/v and output projections with bias. Either `nn.MultiheadAttention(D, num_heads, batch_first=True)` or your own (a fused `Linear(D, 3D)` for q, k, v plus `Linear(D, D)` out, or four `Linear(D, D)`); no mask, no positional encoding inside the block;
   - MLP: `Linear(D, int(D * mlp_ratio)) → GELU(approximate="tanh") → Linear(int(D * mlp_ratio), D)`;
   - `self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(D, 6 * D))` with the Linear's **weight and bias initialized to zero**, chunked (`.chunk(6, dim=1)`) in the order `shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp`;
   - the two gated pre-norm residual updates above.
4. **`patchify(images, patch_size)`** — `(B, C, H, W)` → `(B, (H/p)(W/p), p·p·C)`. Patches in **row-major** order (token $k = i \cdot (W/p) + j$ holds pixels `[i·p:(i+1)·p, j·p:(j+1)·p]`); inside a token the features are ordered **(row-in-patch, col-in-patch, channel)** — the MAE / DiT layout.
5. **`unpatchify(tokens, patch_size, channels, h, w)`** — exact inverse of `patchify`; `h`, `w` are the image height and width **in pixels**.

---

### Constraints

- ✅ `nn.MultiheadAttention` (with `batch_first=True`) or `F.scaled_dot_product_attention` are fine.
- ✅ Assume `H` and `W` are divisible by `patch_size` and `hidden_size` by `num_heads`.
- ❌ No `timm`, `diffusers` or other model zoos.
- ❌ No learnable affine in the block's LayerNorms — adaLN supplies the scale and shift.
- ❌ No positional embedding or attention mask inside the block.

---

<details>
  <summary>💡 Hint</summary>

  - The conditioning is per *sample* but the activations are per *token*: every `(B, D)` modulation vector needs `unsqueeze(1)` to become `(B, 1, D)`.
  - Why `1 + scale`? With zero-initialized adaLN, `scale = 0` must leave the normalized activations untouched.
  - Sanity check: at initialization your block should return `x` **exactly**, whatever `c` is.
  - `patchify` is one `reshape` → `permute` → `reshape`; `unpatchify` runs the same three steps backwards. Write down the axis order `(B, C, H/p, p, W/p, p)` before permuting.

</details>

---

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

In [ ]:
def modulate(x, shift, scale):
    # (B, D) -> (B, 1, D): one shift/scale per sample, shared by all its tokens
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


class TimestepEmbedder(nn.Module):
    def __init__(self, hidden_size, freq_dim=256):
        super().__init__()
        self.freq_dim = freq_dim
        self.mlp = nn.Sequential(
            nn.Linear(freq_dim, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
        )

    @staticmethod
    def timestep_embedding(t, dim, max_period=10000):
        half = dim // 2
        freqs = torch.exp(-math.log(max_period)
                          * torch.arange(half, dtype=torch.float32, device=t.device) / half)
        args = t.float()[:, None] * freqs[None, :]               # (N, half): periods 2*pi .. 2*pi*max_period
        emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        if dim % 2:                                              # odd dim: pad one zero column
            emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=-1)
        return emb

    def forward(self, t):
        return self.mlp(self.timestep_embedding(t, self.freq_dim))


class Attention(nn.Module):
    """Multi-head self-attention with a fused qkv projection (the timm / DiT layout)."""

    def __init__(self, dim, num_heads):
        super().__init__()
        assert dim % num_heads == 0, "hidden_size must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)            # each (B, heads, N, head_dim)
        attn = (q @ k.transpose(-2, -1)) * self.head_dim ** -0.5  # (B, heads, N, N)
        out = attn.softmax(dim=-1) @ v                            # (B, heads, N, head_dim)
        return self.proj(out.transpose(1, 2).reshape(B, N, D))


class DiTBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.attn = Attention(hidden_size, num_heads)
        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        mlp_hidden = int(hidden_size * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, mlp_hidden),
            nn.GELU(approximate="tanh"),
            nn.Linear(mlp_hidden, hidden_size),
        )
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(hidden_size, 6 * hidden_size))
        # adaLN-Zero: all six vectors start at 0 -> gates are 0 -> the block is the identity
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)

    def forward(self, x, c):
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = \
            self.adaLN_modulation(c).chunk(6, dim=1)               # each (B, D)
        x = x + gate_msa.unsqueeze(1) * self.attn(modulate(self.norm1(x), shift_msa, scale_msa))
        x = x + gate_mlp.unsqueeze(1) * self.mlp(modulate(self.norm2(x), shift_mlp, scale_mlp))
        return x


def patchify(images, patch_size):
    B, C, H, W = images.shape
    p = patch_size
    x = images.reshape(B, C, H // p, p, W // p, p)             # split H and W into (grid, within-patch)
    x = x.permute(0, 2, 4, 3, 5, 1)                            # (B, H/p, W/p, p, p, C)
    return x.reshape(B, (H // p) * (W // p), p * p * C)        # row-major tokens, (row, col, ch) features


def unpatchify(tokens, patch_size, channels, h, w):
    B = tokens.shape[0]
    p = patch_size
    x = tokens.reshape(B, h // p, w // p, p, p, channels)      # undo the last reshape
    x = x.permute(0, 5, 1, 3, 2, 4)                            # (B, C, H/p, p, W/p, p)
    return x.reshape(B, channels, h, w)

In [ ]:
# Test 1: modulate and the timestep embedder
x, shift, scale = torch.randn(2, 3, 4), torch.randn(2, 4), torch.randn(2, 4)
assert torch.allclose(modulate(x, shift, scale), x * (1 + scale[:, None]) + shift[:, None])

t = torch.tensor([0.0, 1.0, 250.5, 999.0])
emb = TimestepEmbedder.timestep_embedding(t, 8)
args = t[:, None] * torch.exp(-math.log(10000) * torch.arange(4) / 4)
assert emb.shape == (4, 8) and torch.allclose(emb, torch.cat([args.cos(), args.sin()], dim=-1), atol=1e-5)
te = TimestepEmbedder(32)
out = te(t)
assert out.shape == (4, 32) and not torch.allclose(out[0], out[1]), "different t -> different embeddings"
assert te(torch.tensor([3, 7])).shape == (2, 32), "integer timesteps must work"
print("Test 1 passed: modulate, timestep embedding")

In [ ]:
# Test 2: adaLN-Zero - identity at init, conditioning matters once the gates open, no positional info
torch.manual_seed(0)
blk = DiTBlock(32, num_heads=4)
x, c = torch.randn(2, 6, 32), torch.randn(2, 32)
assert torch.allclose(blk(x, c), x, atol=1e-6), "a freshly initialised DiT block must be the identity"

with torch.no_grad():                                   # pretend training moved the adaLN weights
    blk.adaLN_modulation[-1].weight.normal_(std=0.2)
    blk.adaLN_modulation[-1].bias.normal_(std=0.2)
out = blk(x, c)
assert out.shape == (2, 6, 32)
assert not torch.allclose(out, blk(x, c.flip(0)), atol=1e-4), "the output must depend on c"
perm = torch.randperm(6)
assert torch.allclose(blk(x[:, perm], c), out[:, perm], atol=1e-5), "the block must be permutation equivariant"
print("Test 2 passed: identity at init, conditioned, permutation equivariant")

In [ ]:
# Test 3: patchify / unpatchify
imgs = torch.randn(2, 3, 8, 12)
tok = patchify(imgs, 4)
assert tok.shape == (2, 6, 48), tok.shape
patch_11 = tok[:, 1 * 3 + 1].reshape(2, 4, 4, 3).permute(0, 3, 1, 2)   # token for patch row 1, col 1
assert torch.equal(patch_11, imgs[:, :, 4:8, 4:8]), "row-major patches, (row, col, channel) features"
assert torch.equal(unpatchify(tok, 4, 3, 8, 12), imgs), "unpatchify must invert patchify"
print("Test 3 passed: patchify / unpatchify")

In [ ]:
# Putting it together: a tiny DiT on 4-channel 16x16 "latents" (patch 2 -> 64 tokens)
class FinalLayer(nn.Module):
    """DiT's output head: adaLN shift/scale + a zero-initialised Linear to p*p*C_out."""

    def __init__(self, hidden, patch, out_ch):
        super().__init__()
        self.norm = nn.LayerNorm(hidden, elementwise_affine=False, eps=1e-6)
        self.linear = nn.Linear(hidden, patch * patch * out_ch)
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(hidden, 2 * hidden))
        for m in (self.linear, self.adaLN_modulation[-1]):
            nn.init.zeros_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x, c):
        shift, scale = self.adaLN_modulation(c).chunk(2, dim=1)
        return self.linear(modulate(self.norm(x), shift, scale))


class TinyDiT(nn.Module):
    def __init__(self, in_ch=4, img=16, patch=2, hidden=64, depth=2, heads=4, num_classes=10):
        super().__init__()
        self.patch, self.in_ch, self.img = patch, in_ch, img
        self.x_embed = nn.Linear(patch * patch * in_ch, hidden)       # = Conv2d(k=s=patch) on patches
        self.pos = nn.Parameter(0.02 * torch.randn(1, (img // patch) ** 2, hidden))  # DiT: fixed 2-D sin-cos
        self.t_embed = TimestepEmbedder(hidden)
        self.y_embed = nn.Embedding(num_classes, hidden)
        self.blocks = nn.ModuleList(DiTBlock(hidden, heads) for _ in range(depth))
        self.final = FinalLayer(hidden, patch, in_ch)

    def forward(self, x, t, y):
        h = self.x_embed(patchify(x, self.patch)) + self.pos          # (B, N, hidden)
        c = self.t_embed(t) + self.y_embed(y)                          # (B, hidden): conditioning
        for blk in self.blocks:
            h = blk(h, c)
        return unpatchify(self.final(h, c), self.patch, self.in_ch, self.img, self.img)


torch.manual_seed(0)
net = TinyDiT()
x = torch.randn(8, 4, 16, 16)
t = torch.randint(0, 1000, (8,))
y = torch.randint(0, 10, (8,))
out = net(x, t, y)
print(f"output {tuple(out.shape)}, {sum(p.numel() for p in net.parameters()):,} parameters")
assert out.shape == x.shape and torch.all(out == 0), "an untrained DiT predicts exactly 0"

# Gradients reach the network progressively: at step 0 only the zero-initialised output layer
# gets a gradient; once it is non-zero, the blocks' adaLN gates start to receive one.
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
target = torch.randn_like(x)
for step in range(3):
    opt.zero_grad()
    F.mse_loss(net(x, t, y), target).backward()
    g_out = net.final.linear.weight.grad.abs().max().item()
    g_gate = net.blocks[0].adaLN_modulation[-1].weight.grad.abs().max().item()
    print(f"step {step}: max |grad| output layer {g_out:.1e}, block-0 adaLN {g_gate:.1e}")
    opt.step()

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("dit-block", modulate=modulate, TimestepEmbedder=TimestepEmbedder, DiTBlock=DiTBlock,
      patchify=patchify, unpatchify=unpatchify)

## Why it works

**adaLN as conditioning.** LayerNorm normalizes each token; the affine parameters that would normally be learned constants are instead *functions of $c$*. Every token of a sample gets the same shift/scale, so the conditioning acts like FiLM: it re-weights and re-centres features globally, which is cheap (one $D \times 6D$ matmul per sample, not per token) and effective for global signals such as the noise level and the class.

**The "Zero".** With the adaLN Linear zero-initialized, $\alpha_1 = \alpha_2 = 0$, so $x \mapsto x + 0 \cdot \text{branch} = x$: a deep stack starts as the identity, the residual stream is not perturbed by random attention/MLP outputs, and the network grows its function from there (the same idea as zero-init of the last BN γ in ResNets / Fixup / ReZero). Learning is not blocked: $\partial\,\text{out}/\partial \alpha = \text{branch}(\cdot) \ne 0$, so the gates get a gradient on the very first step. DiT additionally zero-initializes the final output Linear, so an untrained model predicts exactly 0 (the demo shows the gradient then reaches the block gates from the second step on).

**Pre-norm + gates.** Each branch reads $\mathrm{LN}(x)$ and writes back through a gate, so the residual stream itself is never normalized — the standard pre-norm recipe that keeps deep transformers trainable, here with a learned, input-dependent residual scale.

**Patchify.** A `(B, C, H, W)` latent becomes $N = HW/p^2$ tokens of size $p^2 C$; DiT then applies a Linear (equivalently a `Conv2d` with kernel = stride = $p$) and adds fixed 2-D sin-cos positional embeddings. Only the order of axes matters, and `unpatchify` must use the same order as the model's output head.

## Complexity

Per block: attention $O(B N^2 D)$ + projections/MLP $O(B N D^2 (4 + 2r))$ + adaLN $O(B D^2 \cdot 6)$ (once per sample). Since $N = HW/p^2$, halving the patch size multiplies the token count by 4 — the main compute knob in DiT. Memory is dominated by the $B \cdot \text{heads} \cdot N^2$ attention matrix unless a fused (Flash) attention kernel is used.

## Common bugs

- **Forgetting `unsqueeze(1)`** on shift/scale/gate: `(B, D)` broadcasts against `(N, D)` — it crashes if $B \ne N$ and silently mixes samples with tokens if $B = N$.
- **`x * scale + shift`** instead of `x * (1 + scale) + shift`: with zero init that zeroes the branch input, and the scale has no neutral value.
- **Not zero-initializing** (or zeroing only the weight, not the bias) — the block is not the identity at init.
- **Wrong chunk order** vs. the reference checkpoint — loads fine, behaves wrongly.
- **`nn.MultiheadAttention` without `batch_first=True`** on `(B, N, D)` input: attention runs over the batch axis.
- **Affine LayerNorm** (redundant parameters that fight adaLN) or **post-norm** placement.
- **Patch layout mismatches** — `(C, p, p)` vs `(p, p, C)` features, or column-major patches — break `unpatchify` against a trained output head.

## Interview follow-ups

1. **Why did adaLN-Zero beat in-context conditioning and cross-attention in DiT?** In the DiT ablation (DiT-XL/2, ImageNet 256), adaLN-Zero gave the best FID at the lowest cost: cross-attention adds ~15% Gflops, in-context tokens add sequence length, and plain adaLN (without zero init) trained worse. For a single global vector (timestep + class), modulating every LayerNorm is the cheapest strong signal; cross-attention pays off when the condition is a *sequence* (text).
2. **Why does zero-initializing the gates help?** Each residual block starts as the identity, so a deep network initially is a well-conditioned shallow one; signal and gradient magnitudes are controlled without warm-up tricks, and blocks "switch on" as the gates grow. The same reasoning motivates zero-init of the last layer of each residual branch (Goyal et al.; Fixup; ReZero) and of ControlNet's zero convolutions.
3. **How does DiT scale?** FID improves smoothly with forward Gflops — whether from more depth/width or more tokens (smaller patches) — and is much better predicted by Gflops than by parameter count; DiT-XL/2 (~119 Gflops) reached FID 2.27 on ImageNet 256 with classifier-free guidance. Larger models are also more *compute-efficient* per training FLOP.
4. **How is text conditioning added in SD3 (MMDiT) / Flux?** Text tokens (T5 + CLIP) and image tokens keep separate weights (two streams with their own QKV/MLP/adaLN) but attend jointly over the concatenated sequence; the pooled CLIP embedding plus the timestep embedding still drive adaLN modulation. Flux follows double-stream MMDiT blocks with cheaper single-stream blocks. PixArt-α instead adds cross-attention to text and shares one adaLN table across blocks (adaLN-single) to cut parameters.
5. **Why does the key bias get no gradient?** Adding a bias to every key adds the same $q \cdot b_k$ to every score in a row, and softmax is shift-invariant, so $\partial L / \partial b_k = 0$ exactly — a nice sanity check when you inspect gradients (many implementations drop the k bias for this reason).
6. **Where do positions come from if the block is permutation-equivariant?** They are added once to the patch embeddings (fixed 2-D sin-cos in DiT; RoPE-2D in newer models such as Flux, which also helps resolution changes). Without them the model could not tell where a patch is.